# Clase 8 — Despliegue en dispositivos y en el navegador: TensorFlow Lite y TensorFlow.js

**Curso:** Fundamentos de TensorFlow desde cero — DIH DATAlife

## Objetivos de aprendizaje

- Explicar cuándo conviene ejecutar un modelo en el dispositivo (edge) en lugar de en un servidor
- Convertir un modelo Keras ya entrenado a TensorFlow Lite y comprobar su tamaño y rendimiento
- Describir cómo se prepara un modelo para ejecutarse en el navegador con TensorFlow.js

En este notebook vas a entrenar un modelo pequeño y rápido sobre Fashion MNIST, convertirlo a TensorFlow Lite (TFLite), comprobar su tamaño en disco y ejecutar una inferencia de prueba con el intérprete de TFLite. Al final verás, en una celda de solo texto, cómo se convertiría ese mismo modelo para ejecutarlo en el navegador con TensorFlow.js (TFJS) — esa parte no se ejecuta aquí porque TensorFlow.js es una librería de JavaScript, no de Python.

Ejecuta este notebook en [Google Colab](https://colab.research.google.com/) con TensorFlow 2.x.

## 1. Preparar los datos y comprobar la versión de TensorFlow

Usamos Fashion MNIST porque ya lo conoces de clases anteriores: son imágenes de 28x28 píxeles en escala de grises, repartidas en diez categorías de ropa. Es un dataset pequeño, así que entrenar un modelo sencillo tarda poco, algo ideal para centrarnos en el proceso de despliegue y no en el entrenamiento.

In [ ]:
import tensorflow as tf
import numpy as np
import os
import time

print("Versión de TensorFlow:", tf.__version__)

In [ ]:
# Cargamos Fashion MNIST directamente desde Keras
fashion_mnist = tf.keras.datasets.fashion_mnist
(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()

# Normalizamos los valores de los píxeles al rango 0-1
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

nombres_clases = [
    "Camiseta", "Pantalón", "Jersey", "Vestido", "Abrigo",
    "Sandalia", "Camisa", "Zapatilla", "Bolso", "Botín"
]

print("Forma de x_train:", x_train.shape)
print("Forma de x_test:", x_test.shape)

## 2. Entrenar un modelo pequeño con Keras

El modelo es deliberadamente sencillo: una capa que aplana la imagen, una capa densa oculta y una capa de salida con softmax para las diez clases. Con pocas épocas es suficiente para nuestro propósito, que es probar el proceso de conversión a TFLite, no conseguir la máxima precisión posible.

In [ ]:
modelo = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=(28, 28)),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(10, activation="softmax")
])

modelo.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

modelo.summary()

In [ ]:
# Entrenamos pocas épocas: el objetivo de este notebook es el despliegue, no la precisión
modelo.fit(x_train, y_train, epochs=5, validation_split=0.1)

In [ ]:
perdida_test, precision_test = modelo.evaluate(x_test, y_test, verbose=0)
print(f"Precisión sobre el conjunto de test: {precision_test:.4f}")

## 3. Convertir el modelo a TensorFlow Lite

Con el modelo ya entrenado, lo convertimos a TFLite usando `tf.lite.TFLiteConverter.from_keras_model`. El resultado es un archivo `.tflite`, un formato binario compacto pensado para cargarse rápido y ocupar poco espacio en un dispositivo móvil o embebido.

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(modelo)
modelo_tflite = converter.convert()

ruta_tflite = "modelo_fashion_mnist.tflite"
with open(ruta_tflite, "wb") as f:
    f.write(modelo_tflite)

print("Modelo convertido y guardado en", ruta_tflite)

## 4. Comprobar el tamaño del archivo en disco

Una de las ventajas de TensorFlow Lite es el tamaño reducido del modelo resultante. Vamos a comparar el tamaño del archivo `.tflite` con el tamaño que ocuparía el modelo Keras original guardado en formato `.h5`, para que veas la diferencia con números reales.

In [ ]:
# Guardamos también el modelo original de Keras para comparar tamaños
ruta_keras = "modelo_fashion_mnist.h5"
modelo.save(ruta_keras)

tamano_tflite_kb = os.path.getsize(ruta_tflite) / 1024
tamano_keras_kb = os.path.getsize(ruta_keras) / 1024

print(f"Tamaño del modelo TFLite:  {tamano_tflite_kb:.1f} KB")
print(f"Tamaño del modelo Keras (.h5): {tamano_keras_kb:.1f} KB")

## 5. Ejecutar una inferencia de prueba con el intérprete de TFLite

El intérprete (`tf.lite.Interpreter`) es el componente que carga el archivo `.tflite` y ejecuta la inferencia. Es el mismo componente que usarías dentro de una aplicación Android o iOS: aquí lo probamos en Python para ver el ciclo completo antes de pasar a la práctica.

In [ ]:
interpreter = tf.lite.Interpreter(model_path=ruta_tflite)
interpreter.allocate_tensors()

detalles_entrada = interpreter.get_input_details()
detalles_salida = interpreter.get_output_details()

print("Forma de entrada esperada:", detalles_entrada[0]["shape"])
print("Forma de salida esperada:", detalles_salida[0]["shape"])

In [ ]:
# Tomamos una imagen de test y le añadimos la dimensión de lote (batch) que espera el modelo
imagen_prueba = x_test[0:1].astype("float32")

interpreter.set_tensor(detalles_entrada[0]["index"], imagen_prueba)
interpreter.invoke()
prediccion_tflite = interpreter.get_tensor(detalles_salida[0]["index"])

clase_predicha = np.argmax(prediccion_tflite[0])
print("Predicción del modelo TFLite:", nombres_clases[clase_predicha])
print("Etiqueta real:", nombres_clases[y_test[0]])

## 6. Comparar el tiempo de inferencia: Keras frente a TFLite

Para hacerte una idea de la diferencia de rendimiento, comparamos cuánto tarda en predecir el modelo Keras original frente al modelo TFLite, sobre el mismo conjunto de imágenes de test.

In [ ]:
num_muestras = 200
lote_prueba = x_test[:num_muestras].astype("float32")

# Tiempo del modelo Keras original
inicio = time.time()
modelo.predict(lote_prueba, verbose=0)
tiempo_keras = time.time() - inicio

# Tiempo del modelo TFLite, imagen a imagen (así es como se usa en un dispositivo real)
inicio = time.time()
for i in range(num_muestras):
    interpreter.set_tensor(detalles_entrada[0]["index"], lote_prueba[i:i+1])
    interpreter.invoke()
    _ = interpreter.get_tensor(detalles_salida[0]["index"])
tiempo_tflite = time.time() - inicio

print(f"Tiempo Keras  ({num_muestras} imágenes): {tiempo_keras:.4f} s")
print(f"Tiempo TFLite ({num_muestras} imágenes): {tiempo_tflite:.4f} s")

Los tiempos concretos dependen de la máquina donde ejecutes el notebook (CPU de Colab, portátil, etc.), así que no te fijes tanto en el número exacto como en la idea: TFLite está optimizado para ejecutar muchas inferencias pequeñas de forma eficiente, que es justo el patrón de uso de una aplicación móvil.

## 7. TensorFlow.js: llevar el modelo al navegador (no ejecutable en este notebook)

TensorFlow.js es una librería de JavaScript, así que esta parte no se ejecuta en un notebook de Python. Aun así, conviene que veas el proceso completo, porque es la tercera pieza del despliegue junto a TFLite.

**Paso 1 — Convertir el modelo Keras a formato TensorFlow.js.** Desde una terminal (no desde este notebook), instalarías la herramienta `tensorflowjs` con `pip install tensorflowjs` y ejecutarías:

```bash
tensorflowjs_converter --input_format=keras \
    modelo_fashion_mnist.h5 \
    modelo_tfjs
```

Esto genera una carpeta `modelo_tfjs` con un archivo `model.json` (la arquitectura del modelo) y uno o varios archivos binarios con los pesos, listos para servirse desde una página web.

**Paso 2 — Cargar y usar el modelo en el navegador.** Este sería el código JavaScript de una página HTML que carga el modelo convertido y clasifica una imagen:

```html
<script src="https://cdn.jsdelivr.net/npm/@tensorflow/tfjs"></script>
<script>
  async function clasificarImagen(tensorImagen) {
    // Cargamos el modelo convertido (equivalente al .tflite, pero para navegador)
    const modelo = await tf.loadLayersModel("modelo_tfjs/model.json");

    // Ejecutamos la inferencia: el equivalente en el navegador a interpreter.invoke()
    const prediccion = modelo.predict(tensorImagen);
    const clase = prediccion.argMax(-1).dataSync()[0];

    console.log("Clase predicha:", clase);
  }
</script>
```

Fíjate en el paralelismo con lo que acabas de hacer en Python: `tf.loadLayersModel` hace el papel de `tf.lite.Interpreter`, y `modelo.predict()` hace el papel de `interpreter.invoke()` + `get_tensor()`. El concepto es el mismo en las tres plataformas (Python, TFLite en móvil, TensorFlow.js en navegador); lo que cambia es el lenguaje y el formato del archivo.